# MULYA — Complete Unified Prototype

One end-to-end notebook: **Lexical → Syntactic → Semantic → MULYA**.

Lexical: BLEU, chrF++, METEOR. Syntactic: POS sequence similarity and dependency-relation F1. Semantic: multilingual BERTScore and COMET. Final: `MULYA = 0.30 LEX + 0.30 SYN + 0.40 SEM`.

In [1]:
!pip -q install sacrebleu nltk stanza bert-score unbabel-comet==2.2.7 tqdm scipy pandas

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import json, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import nltk, stanza, sacrebleu
warnings.filterwarnings("ignore")
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.translate.meteor_score import meteor_score
from sacrebleu.metrics import BLEU, CHRF


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 45.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [2]:
# ============================================================
# CELL 1 — SAFE MULYA ENVIRONMENT SETUP
# ============================================================

# DO NOT install numpy / pandas / scipy manually.
# Colab already provides compatible versions of these.

!pip install -q \
    sacrebleu \
    nltk \
    stanza \
    bert-score \
    unbabel-comet \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 74.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ============================================================
# CELL 2 — ENVIRONMENT VERIFICATION
# ============================================================

import numpy as np
import pandas as pd
import scipy
import torch
import sacrebleu
import stanza
import nltk
import bert_score

print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("SciPy      :", scipy.__version__)
print("PyTorch    :", torch.__version__)
print("SacreBLEU  :", sacrebleu.__version__)
print("Stanza     :", stanza.__version__)
print("NLTK       :", nltk.__version__)
print("GPU        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device :", torch.cuda.get_device_name(0))

print("\n✅ MULYA environment verification completed.")

NumPy      : 1.26.4
Pandas     : 2.2.3
SciPy      : 1.16.3
PyTorch    : 2.11.0+cpu
SacreBLEU  : 2.6.0
Stanza     : 1.15.0
NLTK       : 3.9.1
GPU        : False

✅ MULYA environment verification completed.


In [2]:
import pandas as pd
from pathlib import Path

print("Pandas loaded:", pd.__version__)

Pandas loaded: 2.2.3


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

EVAL_PATH = PROJECT_ROOT / "processed/evaluation_ready/unified_evaluation_data.csv"

print("Looking for:", EVAL_PATH)

if not EVAL_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {EVAL_PATH}")

df = pd.read_csv(EVAL_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Looking for: /content/drive/MyDrive/UniMTScore/processed/evaluation_ready/unified_evaluation_data.csv
Dataset loaded successfully!
Shape: (42288, 9)
Columns: ['evaluation_id', 'dataset', 'subset', 'source', 'reference', 'translation', 'source_language', 'target_language', 'direction']


,evaluation_id,dataset,subset,source,reference,translation,source_language,target_language,direction
0,1,FLORES-200,dev,"On Monday, scientists from the Stanford Univer...","સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...","સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...",English,Gujarati,eng_Latn→guj_Gujr
1,2,FLORES-200,dev,Lead researchers say this may bring early dete...,"અગ્રણી સંશોધનકર્તાઓ કહે છે કે, સમૃદ્ધ દેશોની સ...","અગ્રણી સંશોધકો કહે છે કે આ કેન્સર, ક્ષય રોગ, એ...",English,Gujarati,eng_Latn→guj_Gujr
2,3,FLORES-200,dev,The JAS 39C Gripen crashed onto a runway at ar...,JAS 39C ગ્રીપન સવારે 9:30 ના સ્થાનિક સમયે (02...,જેએએસ 39 સી ગ્રીપેન સ્થાનિક સમયानुसार સવારે 9:...,English,Gujarati,eng_Latn→guj_Gujr
3,4,FLORES-200,dev,The pilot was identified as Squadron Leader Di...,પાઇલટની ઓળખ સ્ક્વૉડ્રન લીડર ડિલોક્રિટ પૅટ્ટાવી...,પાયલોટની ઓળખ એ સ્ક્વોડ્રન લીડર દિલોકૃત પટ્ટાવી...,English,Gujarati,eng_Latn→guj_Gujr
4,5,FLORES-200,dev,Local media reports an airport fire vehicle ro...,"સ્થાનિક મીડિયાના અહેવાલો મુજબ, જવાબ આપતી વખતે ...",સ્થાનિક મીડિયા અહેવાલો એરપોર્ટ ફાયર વાહન પ્રતિ...,English,Gujarati,eng_Latn→guj_Gujr


In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")
EVAL_PATH = PROJECT_ROOT / "processed/evaluation_ready/unified_evaluation_data.csv"
LEXICAL_DIR = PROJECT_ROOT / "features/lexical"
SYNTACTIC_DIR = PROJECT_ROOT / "features/syntactic"
SEMANTIC_DIR = PROJECT_ROOT / "features/semantic"
RESULTS_DIR = PROJECT_ROOT / "results/MULYA"
for p in [LEXICAL_DIR,SYNTACTIC_DIR,SEMANTIC_DIR,RESULTS_DIR]: p.mkdir(parents=True, exist_ok=True)

if not EVAL_PATH.exists(): raise FileNotFoundError(EVAL_PATH)
df = pd.read_csv(EVAL_PATH)
rename = {}
if "mt" in df.columns and "translation" not in df.columns: rename["mt"]="translation"
if "hypothesis" in df.columns and "translation" not in df.columns: rename["hypothesis"]="translation"
if "ref" in df.columns and "reference" not in df.columns: rename["ref"]="reference"
if "target" in df.columns and "reference" not in df.columns: rename["target"]="reference"
if "src" in df.columns and "source" not in df.columns: rename["src"]="source"
df=df.rename(columns=rename)
for x in ["source","reference","translation"]:
    if x not in df.columns: raise ValueError(f"Missing required column: {x}")
if "evaluation_id" not in df.columns: df.insert(0,"evaluation_id",np.arange(1,len(df)+1))
df["evaluation_id"]=df["evaluation_id"].astype(str)
if df.evaluation_id.duplicated().any(): raise ValueError("Duplicate evaluation_id")
print("Unified dataset:",df.shape)
print(df.columns.tolist())


Unified dataset: (42288, 9)
['evaluation_id', 'dataset', 'subset', 'source', 'reference', 'translation', 'source_language', 'target_language', 'direction']


In [8]:
# ============================================================
# LEXICAL METRIC IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sacrebleu.metrics import BLEU, CHRF

print("SacreBLEU imported successfully.")
print("BLEU:", BLEU)
print("chrF++:", CHRF)

SacreBLEU imported successfully.
BLEU: <class 'sacrebleu.metrics.bleu.BLEU'>
chrF++: <class 'sacrebleu.metrics.chrf.CHRF'>


In [17]:
from tqdm.auto import tqdm

print("✅ tqdm loaded")

✅ tqdm loaded


## 1. Lexical pipeline

In [11]:
bleu_metric = BLEU(effective_order=True)
chrfpp_metric = CHRF(word_order=2)

def bleu(ref, mt):
    if pd.isna(ref) or pd.isna(mt):
        return np.nan

    ref = str(ref).strip()
    mt = str(mt).strip()

    if not ref or not mt:
        return np.nan

    return bleu_metric.sentence_score(mt, [ref]).score / 100.0


def chrfpp(ref, mt):
    if pd.isna(ref) or pd.isna(mt):
        return np.nan

    ref = str(ref).strip()
    mt = str(mt).strip()

    if not ref or not mt:
        return np.nan

    return chrfpp_metric.sentence_score(mt, [ref]).score / 100.0

In [13]:
# ============================================================
# COMPLETE LEXICAL IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sacrebleu.metrics import BLEU, CHRF

from nltk.translate.meteor_score import meteor_score

print("✅ NumPy loaded")
print("✅ Pandas loaded")
print("✅ SacreBLEU BLEU loaded")
print("✅ SacreBLEU chrF++ loaded")
print("✅ NLTK METEOR loaded")

✅ NumPy loaded
✅ Pandas loaded
✅ SacreBLEU BLEU loaded
✅ SacreBLEU chrF++ loaded
✅ NLTK METEOR loaded


In [14]:
def meteor(ref, mt):
    if pd.isna(ref) or pd.isna(mt):
        return np.nan

    ref = str(ref).strip()
    mt = str(mt).strip()

    if not ref or not mt:
        return np.nan

    return meteor_score(
        [ref.split()],
        mt.split()
    )

In [19]:
# ============================================================
# NLTK RESOURCES REQUIRED FOR METEOR
# ============================================================

import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

print("✅ NLTK WordNet resources downloaded.")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✅ NLTK WordNet resources downloaded.


In [21]:
bleu_metric = BLEU(effective_order=True)
chrfpp_metric = CHRF(word_order=2)

df["BLEU"] = df.apply(
    lambda r: bleu(r["reference"], r["translation"]),
    axis=1
)

df["chrF++"] = df.apply(
    lambda r: chrfpp(r["reference"], r["translation"]),
    axis=1
)

df["METEOR"] = df.apply(
    lambda r: meteor(r["reference"], r["translation"]),
    axis=1
)

print("✅ Lexical evaluation completed.")

display(
    df[
        ["source", "reference", "translation",
         "BLEU", "chrF++", "METEOR"]
    ].head()
)

✅ Lexical evaluation completed.


,source,reference,translation,BLEU,chrF++,METEOR
0,"On Monday, scientists from the Stanford Univer...","સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...","સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...",0.326358,0.583828,0.576039
1,Lead researchers say this may bring early dete...,"અગ્રણી સંશોધનકર્તાઓ કહે છે કે, સમૃદ્ધ દેશોની સ...","અગ્રણી સંશોધકો કહે છે કે આ કેન્સર, ક્ષય રોગ, એ...",0.098971,0.412659,0.309569
2,The JAS 39C Gripen crashed onto a runway at ar...,JAS 39C ગ્રીપન સવારે 9:30 ના સ્થાનિક સમયે (02...,જેએએસ 39 સી ગ્રીપેન સ્થાનિક સમયानुसार સવારે 9:...,0.101443,0.321637,0.183752
3,The pilot was identified as Squadron Leader Di...,પાઇલટની ઓળખ સ્ક્વૉડ્રન લીડર ડિલોક્રિટ પૅટ્ટાવી...,પાયલોટની ઓળખ એ સ્ક્વોડ્રન લીડર દિલોકૃત પટ્ટાવી...,0.047892,0.401762,0.180723
4,Local media reports an airport fire vehicle ro...,"સ્થાનિક મીડિયાના અહેવાલો મુજબ, જવાબ આપતી વખતે ...",સ્થાનિક મીડિયા અહેવાલો એરપોર્ટ ફાયર વાહન પ્રતિ...,0.038957,0.380970,0.157480


In [25]:
bleu_metric=BLEU(effective_order=True)
chrfpp_metric=CHRF(word_order=2)

def bleu(ref, mt):
    if pd.isna(ref) or pd.isna(mt):
        return np.nan

    ref = str(ref).strip()
    mt = str(mt).strip()

    if not ref or not mt:
        return np.nan

    score = bleu_metric.sentence_score(
        mt,
        [ref]
    ).score

    return score / 100.0

def chrfpp(ref,mt):
    if pd.isna(ref) or pd.isna(mt) or not str(ref).strip() or not str(mt).strip(): return np.nan
    return chrfpp_metric.sentence_score(str(mt).strip(),[str(ref).strip()]).score/100.0

def meteor(ref,mt):
    if pd.isna(ref) or pd.isna(mt): return np.nan
    r=str(ref).split(); m=str(mt).split()
    if not r or not m: return np.nan
    return meteor_score([r],m)

tqdm.pandas(desc="BLEU")
df["bleu_score"]=df.progress_apply(lambda r:bleu(r.reference,r.translation),axis=1)
tqdm.pandas(desc="chrF++")
df["chrfpp_score"]=df.progress_apply(lambda r:chrfpp(r.reference,r.translation),axis=1)
tqdm.pandas(desc="METEOR")
df["meteor_score"]=df.progress_apply(lambda r:meteor(r.reference,r.translation),axis=1)
df["lexical_score"]=0.30*df.bleu_score+0.35*df.chrfpp_score+0.35*df.meteor_score


BLEU:   0%|          | 0/42288 [00:00<?, ?it/s]

chrF++:   0%|          | 0/42288 [00:00<?, ?it/s]

METEOR:   0%|          | 0/42288 [00:00<?, ?it/s]

In [26]:
from tqdm.auto import tqdm

tqdm.pandas(desc="Recalculating BLEU")

df["bleu_score"] = df.progress_apply(
    lambda r: bleu(r["reference"], r["translation"]),
    axis=1
)

print("BLEU recalculated.")
print("Minimum:", df["bleu_score"].min())
print("Maximum:", df["bleu_score"].max())

Recalculating BLEU:   0%|          | 0/42288 [00:00<?, ?it/s]

BLEU recalculated.
Minimum: 0.0
Maximum: 1.0000000000000004


In [28]:
# ============================================================
# DIAGNOSE BLEU VALUES
# ============================================================

s = df["bleu_score"]

print("dtype:", s.dtype)
print("count:", len(s))
print("NaN:", s.isna().sum())
print("inf:", np.isinf(s).sum())

print("\nMinimum:", s.min())
print("Maximum:", s.max())

print("\nValues outside [0,1]:")
bad = df.loc[
    s.notna() & ~s.between(0, 1),
    ["reference", "translation", "bleu_score"]
]

print("Number of bad rows:", len(bad))
display(bad.head(20))

dtype: float64
count: 42288
NaN: 0
inf: 0

Minimum: 0.0
Maximum: 1.0000000000000004

Values outside [0,1]:
Number of bad rows: 579


,reference,translation,bleu_score
664,વોલ્ટ ડિઝની વર્લ્ડની મુસાફરી ઘણા અમેરિકન પરિવા...,વોલ્ટ ડિઝની વર્લ્ડની મુસાફરી ઘણા અમેરિકન પરિવા...,1.0
4295,मंगळवारी ओसाकामध्ये त्यांचा मृत्यू झाला.,मंगळवारी ओसाकामध्ये त्यांचा मृत्यू झाला.,1.0
5235,ହଷ୍ଟେଲର ଅତିଥିମାନେ ଅଧିକାଂଶ ସଂଯୁକ୍ତ ଆରବ ଏମିରେଟ୍ସ...,ହଷ୍ଟେଲର ଅତିଥିମାନେ ଅଧିକାଂଶ ସଂଯୁକ୍ତ ଆରବ ଏମିରେଟ୍ସ...,1.0
6006,He had 2 goals and 2 assists in Washington's 5...,He had 2 goals and 2 assists in Washington's 5...,1.0
6176,The Brazilian suffered a serious head injury a...,The Brazilian suffered a serious head injury a...,1.0
6232,The hostel's guests were mostly citizens of th...,The hostel's guests were mostly citizens of th...,1.0
6260,Former U.S. Speaker of the House Newt Gingrich...,Former U.S. Speaker of the House Newt Gingrich...,1.0
6311,"During his time with the team, he scored 403 g...","During his time with the team, he scored 403 g...",1.0
6452,The balance of power was a system in which Eur...,The balance of power was a system in which Eur...,1.0
6576,Animals are found all over the earth. They dig...,Animals are found all over the earth. They dig...,1.0


In [29]:
import inspect

print(inspect.getsource(bleu))

def bleu(ref, mt):
    if pd.isna(ref) or pd.isna(mt):
        return np.nan

    ref = str(ref).strip()
    mt = str(mt).strip()

    if not ref or not mt:
        return np.nan

    score = bleu_metric.sentence_score(
        mt,
        [ref]
    ).score

    return score / 100.0



In [30]:
def bleu(ref, mt):
    if pd.isna(ref) or pd.isna(mt):
        return np.nan

    ref = str(ref).strip()
    mt = str(mt).strip()

    if not ref or not mt:
        return np.nan

    score = bleu_metric.sentence_score(
        mt,
        [ref]
    ).score

    return score / 100.0

In [31]:
df["bleu_score"] = np.nan

tqdm.pandas(desc="BLEU")

df["bleu_score"] = df.progress_apply(
    lambda r: bleu(
        r["reference"],
        r["translation"]
    ),
    axis=1
)

print(
    "BLEU range:",
    df["bleu_score"].min(),
    "to",
    df["bleu_score"].max()
)

BLEU:   0%|          | 0/42288 [00:00<?, ?it/s]

BLEU range: 0.0 to 1.0000000000000004


In [32]:
lex_cols = [
    "bleu_score",
    "chrfpp_score",
    "meteor_score",
    "lexical_score"
]

for col in lex_cols:
    if df[col].isna().any():
        raise ValueError(
            f"Lexical validation failed: {col} contains NaN"
        )

    if not df[col].between(0, 1).all():
        raise ValueError(
            f"Lexical validation failed: {col} contains values outside [0,1]"
        )

print("✅ All lexical scores are valid in [0,1].")

print(
    df[lex_cols]
    .describe()
    .round(4)
)

ValueError: Lexical validation failed: bleu_score contains values outside [0,1]

In [24]:
# ============================================================
# DEBUG BLEU SCALE
# ============================================================

print("BLEU min:", df["bleu_score"].min())
print("BLEU max:", df["bleu_score"].max())

print("\nNaN count:", df["bleu_score"].isna().sum())

print("\nValues > 1:")
display(
    df.loc[
        df["bleu_score"] > 1,
        ["reference", "translation", "bleu_score"]
    ].head(10)
)

print("\nValues < 0:")
display(
    df.loc[
        df["bleu_score"] < 0,
        ["reference", "translation", "bleu_score"]
    ].head(10)
)

BLEU min: 0.0
BLEU max: 1.0000000000000004

NaN count: 0

Values > 1:


,reference,translation,bleu_score
664,વોલ્ટ ડિઝની વર્લ્ડની મુસાફરી ઘણા અમેરિકન પરિવા...,વોલ્ટ ડિઝની વર્લ્ડની મુસાફરી ઘણા અમેરિકન પરિવા...,1.0
4295,मंगळवारी ओसाकामध्ये त्यांचा मृत्यू झाला.,मंगळवारी ओसाकामध्ये त्यांचा मृत्यू झाला.,1.0
5235,ହଷ୍ଟେଲର ଅତିଥିମାନେ ଅଧିକାଂଶ ସଂଯୁକ୍ତ ଆରବ ଏମିରେଟ୍ସ...,ହଷ୍ଟେଲର ଅତିଥିମାନେ ଅଧିକାଂଶ ସଂଯୁକ୍ତ ଆରବ ଏମିରେଟ୍ସ...,1.0
6006,He had 2 goals and 2 assists in Washington's 5...,He had 2 goals and 2 assists in Washington's 5...,1.0
6176,The Brazilian suffered a serious head injury a...,The Brazilian suffered a serious head injury a...,1.0
6232,The hostel's guests were mostly citizens of th...,The hostel's guests were mostly citizens of th...,1.0
6260,Former U.S. Speaker of the House Newt Gingrich...,Former U.S. Speaker of the House Newt Gingrich...,1.0
6311,"During his time with the team, he scored 403 g...","During his time with the team, he scored 403 g...",1.0
6452,The balance of power was a system in which Eur...,The balance of power was a system in which Eur...,1.0
6576,Animals are found all over the earth. They dig...,Animals are found all over the earth. They dig...,1.0



Values < 0:


,reference,translation,bleu_score


In [33]:
lex_cols=["bleu_score","chrfpp_score","meteor_score","lexical_score"]
for col in lex_cols:
    if df[col].isna().any() or not df[col].between(0,1).all(): raise ValueError(f"Lexical validation failed: {col}")
print(df[lex_cols].describe().round(4))
df[[c for c in ["evaluation_id","dataset","subset","source_language","target_language","direction","bleu_score","chrfpp_score","meteor_score","lexical_score"] if c in df.columns]].to_csv(LEXICAL_DIR/"mulya_lexical_scores.csv",index=False,encoding="utf-8-sig")


ValueError: Lexical validation failed: bleu_score

## 2. Syntactic pipeline
The implementation follows the existing project logic: `SequenceMatcher` for POS sequences and F1 overlap of dependency-relation labels. The parser is selected by **target language**, which is required for both forward and reverse directions.

In [ ]:
LANG_CODES={"English":"en","Hindi":"hi","Gujarati":"gu","Kannada":"kn","Malayalam":"ml","Marathi":"mr","Odia":"or"}
ALIASES={"en":"English","eng":"English","English":"English","hi":"Hindi","hin":"Hindi","Hindi":"Hindi","gu":"Gujarati","Gujarati":"Gujarati","kn":"Kannada","Kannada":"Kannada","ml":"Malayalam","Malayalam":"Malayalam","mr":"Marathi","Marathi":"Marathi","or":"Odia","Odia":"Odia","Oriya":"Odia"}
def canon(x): return ALIASES.get(str(x),str(x))

if "target_language" not in df.columns: raise ValueError("target_language required for syntactic evaluation")
df["target_language_canonical"]=df.target_language.map(canon)
needed=sorted(set(df.target_language_canonical))
unsupported=[x for x in needed if x not in LANG_CODES]
if unsupported: raise ValueError(f"Unsupported target languages: {unsupported}")
for lang in sorted(set(LANG_CODES[x] for x in needed)): stanza.download(lang,processors="tokenize,pos,depparse",verbose=False)
pipelines={lang:stanza.Pipeline(lang=code,processors="tokenize,pos,depparse",use_gpu=False,verbose=False) for lang,code in LANG_CODES.items() if lang in needed}
print("Pipelines:",list(pipelines))

def extract(text,pipe):
    doc=pipe("" if pd.isna(text) else str(text)); pos=[]; dep=[]
    for sent in doc.sentences:
        for w in sent.words: pos.append(w.upos); dep.append(w.deprel)
    return pos,dep

def seqsim(a,b): return SequenceMatcher(None,a,b).ratio()
def depsim(a,b):
    if not a and not b:return 1.0
    if not a or not b:return 0.0
    c1,c2=Counter(a),Counter(b); overlap=sum((c1&c2).values()); p=overlap/sum(c2.values()); r=overlap/sum(c1.values())
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def syntax_row(row):
    pipe=pipelines[row.target_language_canonical]
    rp,rd=extract(row.reference,pipe); mp,md=extract(row.translation,pipe)
    return seqsim(rp,mp),depsim(rd,md)

syntax=[]
for _,row in tqdm(df.iterrows(),total=len(df),desc="Syntactic evaluation"):
    syntax.append(syntax_row(row))
df["pos_similarity"]=[x[0] for x in syntax]
df["dependency_similarity"]=[x[1] for x in syntax]
df["syntactic_score"]=0.40*df.pos_similarity+0.60*df.dependency_similarity


In [ ]:
syn_cols=["pos_similarity","dependency_similarity","syntactic_score"]
for col in syn_cols:
    if df[col].isna().any() or not df[col].between(0,1).all(): raise ValueError(f"Syntactic validation failed: {col}")
assert (df.syntactic_score-(0.40*df.pos_similarity+0.60*df.dependency_similarity)).abs().max()<1e-10
df[[c for c in ["evaluation_id","dataset","subset","source_language","target_language","direction","pos_similarity","dependency_similarity","syntactic_score"] if c in df.columns]].to_csv(SYNTACTIC_DIR/"mulya_syntactic_scores.csv",index=False,encoding="utf-8-sig")
print(df[syn_cols].describe().round(4))


## 3. Semantic pipeline
BERTScore uses multilingual XLM-R instead of hard-coding Hindi/English. COMET receives the actual source, MT and reference for every row.

In [ ]:
from bert_score import score as bert_score
BERT_MODEL="xlm-roberta-base"
BATCH=16
df["bertscore_f1"]=np.nan
for start in tqdm(range(0,len(df),BATCH),desc="BERTScore"):
    end=min(start+BATCH,len(df)); part=df.iloc[start:end]
    cand=part.translation.fillna("").astype(str).tolist(); refs=part.reference.fillna("").astype(str).tolist()
    _,_,f1=bert_score(cand,refs,model_type=BERT_MODEL,verbose=False,rescale_with_baseline=False)
    df.loc[df.index[start:end],"bertscore_f1"]=f1.cpu().numpy()
print(df.bertscore_f1.describe())


In [ ]:
from comet import download_model, load_from_checkpoint
COMET_NAME="Unbabel/wmt22-comet-da"
model_path=download_model(COMET_NAME)
comet_model=load_from_checkpoint(model_path)
comet_data=[{"src":"" if pd.isna(r.source) else str(r.source),"mt":"" if pd.isna(r.translation) else str(r.translation),"ref":"" if pd.isna(r.reference) else str(r.reference)} for _,r in df.iterrows()]
output=comet_model.predict(comet_data,batch_size=8,gpus=0)
raw=np.asarray(output.scores,dtype=float)
if raw.min()>=0 and raw.max()<=1: df["comet_score"]=raw; COMET_NORM="unchanged"
elif raw.min()>=-1 and raw.max()<=1: df["comet_score"]=(raw+1)/2; COMET_NORM="[-1,1] to [0,1]"
else: raise ValueError(f"Unexpected COMET range: {raw.min()}..{raw.max()}")
df["semantic_score"]=0.40*df.bertscore_f1+0.60*df.comet_score


In [ ]:
sem_cols=["bertscore_f1","comet_score","semantic_score"]
for col in sem_cols:
    if df[col].isna().any() or not df[col].between(0,1).all(): raise ValueError(f"Semantic validation failed: {col}")
df[[c for c in ["evaluation_id","dataset","subset","source_language","target_language","direction","bertscore_f1","comet_score","semantic_score"] if c in df.columns]].to_csv(SEMANTIC_DIR/"mulya_semantic_scores.csv",index=False,encoding="utf-8-sig")
print(df[sem_cols].describe().round(4))


## 4. Final MULYA fusion

In [ ]:
df["mulya_score"]=0.30*df.lexical_score+0.30*df.syntactic_score+0.40*df.semantic_score
final_cols=["bleu_score","chrfpp_score","meteor_score","lexical_score","pos_similarity","dependency_similarity","syntactic_score","bertscore_f1","comet_score","semantic_score","mulya_score"]
for col in final_cols:
    if df[col].isna().any() or not df[col].between(0,1).all(): raise ValueError(f"Final validation failed: {col}")
assert (df.mulya_score-(0.30*df.lexical_score+0.30*df.syntactic_score+0.40*df.semantic_score)).abs().max()<1e-10
print("FINAL PIPELINE PASSED")
print("Rows:",len(df),"Unique IDs:",df.evaluation_id.nunique())
display(df[["evaluation_id","lexical_score","syntactic_score","semantic_score","mulya_score"]].head(10))


In [ ]:
output_cols=[c for c in ["evaluation_id","dataset","subset","source_language","target_language","direction","source","reference","translation"]+final_cols if c in df.columns]
final_df=df[output_cols].copy()
FINAL_OUTPUT=RESULTS_DIR/"final_mulya_scores.csv"
final_df.to_csv(FINAL_OUTPUT,index=False,encoding="utf-8-sig")

if "dataset" in final_df.columns:
    final_df.groupby("dataset")[final_cols].mean().round(4).to_csv(RESULTS_DIR/"dataset_summary.csv")
if "direction" in final_df.columns:
    final_df.groupby("direction")[final_cols].mean().round(4).to_csv(RESULTS_DIR/"direction_summary.csv")
if "target_language" in final_df.columns:
    final_df.groupby("target_language")[final_cols].mean().round(4).to_csv(RESULTS_DIR/"language_summary.csv")

method={"lexical":{"BLEU":.30,"chrF++":.35,"METEOR":.35},"syntactic":{"POS":.40,"Dependency":.60},"semantic":{"BERTScore_F1":.40,"COMET":.60},"outer":{"LEX":.30,"SYN":.30,"SEM":.40},"bert_model":BERT_MODEL,"comet_model":COMET_NAME,"comet_normalization":COMET_NORM,"dependency_definition":"F1 overlap of dependency-relation label multisets, matching the existing project logic."}
with open(RESULTS_DIR/"mulya_methodology.json","w") as f: json.dump(method,f,indent=2)

print("Saved final output:",FINAL_OUTPUT)
print("Final shape:",final_df.shape)
display(final_df.head())


## Final audit
The saved file is the complete prototype output. Every row contains the component metrics, the three dimensions, and the final MULYA score.